In [1]:
#relevant imports to prepare data
import json
import re
import glob
import os
from collections import defaultdict
import unicodedata


In [2]:
#path to filtered tax ruling json files (path leverages on volume mounting)
path_to_files = "/home/jovyan/data/processed/"

In [3]:
#Normalization functions to strip accents, capitalize characters. 
#Relevant for data preparation
def strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )


def norm_tipo(s: str) -> str:
    return re.sub(r"\s+", " ", strip_accents(s or "")).strip().upper()


In [31]:
#Base Regex to identify the decision section of the ruling.
RE_DECISION_BLOCK = re.compile(
    r"DECISION\b.{0,300}?ADMINISTRANDO\s+JUSTICIA[\s,]+(?:EN\s+)?NOMBRE\s+DEL\s+PUEBLO\s+SOBERANO\s+"
    r"DEL\s+ECUADOR[\s,]*(?:Y\s+)?POR\s+AUTORIDAD\s+DE\s+LA\s+CONSTITUCION\s+Y\s+"
    r"LAS\s+LEYES\s+DE\s+LA\s+REPUBLICA[\s,]*(.*)", 
    re.DOTALL
)
#Regex to match the three types of rulings from the taxpayer's perspective:
#Accept the judicial challenge, deny the judicial challenge or partially accept the judicial challenge. 
RE_PARTIAL = re.compile(
    r"ACEPT\w*\s+PARCIAL|PARCIALMENTE|ACCOGER?\s+PARCIAL"
)

RE_ACCEPT = re.compile(
    r"\b(ACEPT\w*|ACOG\w*)\s+(LA\s+|SU\s+)?DEMANDA"
    r"|\bSE\s+ACEPTA\b"
    r"|\bDECLARAR\s+(LA\s+)?VALIDEZ\b"  
)

RE_DENY = re.compile(
    r"\b(NIEG\w*|NEG\w*|RECHAZ\w*|DESECH\w*)\s+(LA\s+|SU\s+)?DEMANDA"
    r"|\bSIN\s+LUGAR\b"
    r"|\bIMPROCEDENTE\b"
    r"|\bDECLARAR\s+SIN\s+LUGAR\b"
)


In [32]:
#find all jsonfiles in directory
json_files = glob.glob(os.path.join(path_to_files, "*.json"))

In [55]:
#Matching, filtering and taggin section. 
#Here is the basis of the functions that will be used to process several tax rulings (data) and to tag it. 
#Use regex to match the relevant part of the ruling where a decision is made
#Use regex to identify the type of ruling and to register it inside a dictionary.
match_counter = 0
ruling_outcome = {"accept": 0, "partial": 0, "denied": 0, "no matches found": 0}
ruling_classifier_filtered = defaultdict(dict)
decision_section_ruling = 2500

for file_path in json_files:
    with open(file_path, 'rb') as file:
        if file.read(2) != b'[]':
            file_name = os.path.basename(file_path).strip('.json')
            #print(file_name)
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    for item in data:
                        ruling = item.get("actividad")
                        #relevant_ruling = ruling[-2500:]
                        clean_ruling = norm_tipo(ruling)
                        print(f"/n {file_name}:")
                        #print(len(clean_ruling))
                        #print(len(ruling))
                        
                        #print(len(relevant_ruling))
                        match = RE_DECISION_BLOCK.search(clean_ruling)
                        if match:
                            
                            #print(match)
                            match_counter += 1
                            # .strip() cleans up any leading or trailing whitespaces/newlines
                            d = match.group(1)
                            #Normalize and strip accents for ruling outcome matching
                            d = norm_tipo(d)
                            
                            #extract the relevant part of the ruling for further processing
                            group_start = match.start()
                            #print(f"/n {file_name}")
                            #print(ruling[group_start:])
                            #print(group_start)
                            #end_ruling = len(ruling) - (decision_section_ruling - group_start)
                            end_ruling = group_start - (len(clean_ruling) - len(ruling))
                            #print(end_ruling)
                            start_ruling = max(0, group_start - 10000)
                            context_before_ruling = ruling[start_ruling:end_ruling]
                            print(f"Span {start_ruling}------{end_ruling}")
                            # print(f"ruling length {len(delimited_ruling)}")
                            # print(delimited_ruling)
                            #print(d)
                            if RE_PARTIAL.search(d):
                                ruling_outcome["partial"] += 1
                                #print(f"partial {d}")
                                #once confirmed the ruling type:
                                ruling_classifier_filtered["partial"][file_name] = context_before_ruling
                                
                            elif RE_ACCEPT.search(d):
                                ruling_outcome["accept"] += 1
                                #print(f"accept {d}")
                                #once confirmed extract a section of the tax ruling:
                                ruling_classifier_filtered["accept"][file_name] = context_before_ruling
 
                            elif RE_DENY.search(d):
                                ruling_outcome["denied"] += 1
                               #print(f"denied {d}")
                                #once confirmed extract a section of the tax ruling:
                                ruling_classifier_filtered["denied"][file_name] = context_before_ruling

                            else:
                                ruling_outcome["no matches found"] += 1
                                ruling_classifier_filtered["no matches found"][file_name] = context_before_ruling
                        else:
                            print("No match found.")
                except json.JSONDecodeError:
                    print(f"Error: {file_name} is not a valid JSON file.")

/n 17510201900374:
Span 49099------59055
/n 17510201900375:
Span 59271------69289
/n 17510201900371:
Span 163768------173756


In [56]:
print(ruling_outcome)

{'accept': 1, 'partial': 1, 'denied': 1, 'no matches found': 0}


In [60]:
print(ruling_classifier_filtered["accept"])

{'17510201900374': 'ues no fue anunciado como tal, no admitido, pero su referencia es para destacar no solo la insuficiencia de motivación en el valor unitario determinado, sino \xa0además, la falta de prueba de la autoridad demandada que justifique que la determinación del valor no fue practicada “abruptamente” como afirma la actora, donde conforme el inciso segundo del artículo 311 del COGEP corresponde a la administración la prueba de los hechos o actos del contribuyente, de los que se concluya la existencia de obligación tributaria; toda vez y más que, como parte del ejercicio del derecho a la defensa –en este caso de la autoridad demandada- “(...) la prueba o la forma de probar es de riesgo de la misma parte que la aporta o anuncia. Esto se conoce como el principio de la autorresponsabilidad de la prueba” donde además “(…) si la parte procesal no presenta prueba suficiente o si los medios que enuncia no son adecuados, ello es bajo su propio riesgo, pero esto no obsta la defensa de

In [29]:
example = " Habiendo abarcado todos los puntos controvertidos, los suscritos jueces del III tribunal del Tribunal Distrital de lo Contencioso Tributario con sede en el Distrito Metropolitano de Quito, provincia de Pichincha, "

In [30]:
len(example)

214